# Day 2 [Retail + E-commerce + Manufacturing] Data Cleaning + QLoRA Configuration

Welcome to the **Day 2 Pipeline** for developing **Qwen-RetailEcomManufacturing** and **Llama-RetailEcomManufacturing** models.

### Objectives for Today:
1. **Mount GDrive**: Mount Google Drive and make sure dependencies are up to date.
2. **Clean & Normalize Data**: Strip anomalies, standardize formatting, and extract instruction-response pairs.
3. **Train/Val/Test Split**: Split combined datasets into **80% Train, 10% Validation, 10% Test** splits.
4. **LoRA Configurations**: Create PEFT LoRA adapter configs (Rank 16, Alpha 32) and BitsAndBytes configs (4-bit quantization).
5. **Load Check**: Verify both models load and fit cleanly inside the Colab T4 GPU VRAM using 4-bit quantization.

---  
## Step 1: Google Drive Integration & Core Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ensure necessary libraries are installed in this runtime session
!pip install -q transformers datasets accelerate peft bitsandbytes wandb

---  
## Step 2: Initialize Directories & Load Raw Data
We set up local directories and copy the raw datasets we downloaded on Day 1 from Google Drive into our local Colab runtime.

In [ ]:
import os

project_dir = "/content/Retail"

# Detect Google Drive project directory dynamically (supporting Retail or Retail LLM folders)
gdrive_dir = "/content/drive/MyDrive/Retail"
if os.path.exists("/content/drive/MyDrive/Retail LLM"):
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"

# 1. Initialize local directory tree
os.makedirs(project_dir, exist_ok=True)
os.makedirs(os.path.join(project_dir, "data/raw"), exist_ok=True)
os.makedirs(os.path.join(project_dir, "configs"), exist_ok=True)
os.makedirs(os.path.join(project_dir, "src"), exist_ok=True)

# 2. Copy raw data from Google Drive to local disk
print(f"[*] Copying raw data from Google Drive path: {gdrive_dir}...")
!cp -r "{gdrive_dir}/data/raw/"* /content/Retail/data/raw/ 2>/dev/null || true

print("[+] Raw files copied locally.")

---  
## Step 3: Write Configuration & Script Files
We write the configuration JSONs and python modules directly from the notebook cells to keep the execution self-contained.

In [ ]:
# 1. Write configs/qwen_lora_config.json
qwen_config_content = "{\n  \"model_type\": \"qwen\",\n  \"base_model_name_or_path\": \"Qwen/Qwen2.5-7B-Instruct\",\n  \"peft_config\": {\n    \"r\": 16,\n    \"lora_alpha\": 32,\n    \"lora_dropout\": 0.05,\n    \"bias\": \"none\",\n    \"task_type\": \"CAUSAL_LM\",\n    \"target_modules\": [\n      \"q_proj\",\n      \"k_proj\",\n      \"v_proj\",\n      \"o_proj\",\n      \"gate_proj\",\n      \"up_proj\",\n      \"down_proj\"\n    ]\n  },\n  \"quantization_config\": {\n    \"load_in_4bit\": true,\n    \"bnb_4bit_quant_type\": \"nf4\",\n    \"bnb_4bit_use_double_quant\": true,\n    \"bnb_4bit_compute_dtype\": \"bfloat16\"\n  }\n}\n";
with open("/content/Retail/configs/qwen_lora_config.json", "w", encoding="utf-8") as f:
    f.write(qwen_config_content)
print("[+] Created configs/qwen_lora_config.json")

# 2. Write configs/llama_lora_config.json
llama_config_content = "{\n  \"model_type\": \"llama\",\n  \"base_model_name_or_path\": \"meta-llama/Meta-Llama-3-8B-Instruct\",\n  \"peft_config\": {\n    \"r\": 16,\n    \"lora_alpha\": 32,\n    \"lora_dropout\": 0.05,\n    \"bias\": \"none\",\n    \"task_type\": \"CAUSAL_LM\",\n    \"target_modules\": [\n      \"q_proj\",\n      \"k_proj\",\n      \"v_proj\",\n      \"o_proj\",\n      \"gate_proj\",\n      \"up_proj\",\n      \"down_proj\"\n    ]\n  },\n  \"quantization_config\": {\n    \"load_in_4bit\": true,\n    \"bnb_4bit_quant_type\": \"nf4\",\n    \"bnb_4bit_use_double_quant\": true,\n    \"bnb_4bit_compute_dtype\": \"bfloat16\"\n  }\n}\n";
with open("/content/Retail/configs/llama_lora_config.json", "w", encoding="utf-8") as f:
    f.write(llama_config_content)
print("[+] Created configs/llama_lora_config.json")

In [ ]:
# 3. Write src/data_cleaning.py
data_cleaning_code = "import os\nimport json\nimport random\n\ndef clean_text(text):\n    \"\"\"\n    Cleans and normalizes raw text input.\n    \"\"\"\n    if not isinstance(text, str):\n        return \"\"\n    \n    # Strip leading/trailing whitespaces and normalize internal spacing\n    text = \" \".join(text.split())\n    \n    # Standardize curly quotes and apostrophes to straight ones\n    text = text.replace(\"“\", \"\\\"\").replace(\"”\", \"\\\"\").replace(\"‘\", \"'\").replace(\"’\", \"'\")\n    \n    return text\n\ndef clean_and_split_data(raw_data_dir=\"data/raw\", processed_data_dir=\"data/processed\"):\n    \"\"\"\n    Cleans raw JSON data, converts to standard instruction-response pairs, \n    removes duplicates, shuffles deterministically, and splits 80/10/10.\n    \"\"\"\n    os.makedirs(processed_data_dir, exist_ok=True)\n    \n    # Files\n    retail_raw_path = os.path.join(raw_data_dir, \"retail_ecommerce_raw.json\")\n    mfg_raw_path = os.path.join(raw_data_dir, \"manufacturing_raw.json\")\n    \n    combined_pairs = []\n    \n    # Load and clean Retail data\n    print(\"[*] Loading Retail dataset...\")\n    if os.path.exists(retail_raw_path):\n        with open(retail_raw_path, \"r\", encoding=\"utf-8\") as f:\n            retail_data = json.load(f)\n            \n        for row in retail_data:\n            instruction = clean_text(row.get(\"instruction\", \"\"))\n            response = clean_text(row.get(\"response\", \"\"))\n            if instruction and response:\n                combined_pairs.append({\n                    \"instruction\": instruction,\n                    \"response\": response\n                })\n        print(f\"[+] Loaded {len(retail_data)} raw retail items.\")\n    else:\n        print(f\"[-] WARNING: Retail raw data file not found at {retail_raw_path}\")\n\n    # Load and clean Manufacturing data\n    print(\"[*] Loading Manufacturing dataset...\")\n    if os.path.exists(mfg_raw_path):\n        with open(mfg_raw_path, \"r\", encoding=\"utf-8\") as f:\n            mfg_data = json.load(f)\n            \n        for row in mfg_data:\n            instruction = clean_text(row.get(\"instruction\", \"\"))\n            response = clean_text(row.get(\"response\", \"\"))\n            if instruction and response:\n                combined_pairs.append({\n                    \"instruction\": instruction,\n                    \"response\": response\n                })\n        print(f\"[+] Loaded {len(mfg_data)} raw manufacturing items.\")\n    else:\n        print(f\"[-] WARNING: Manufacturing raw data file not found at {mfg_raw_path}\")\n\n    total_loaded = len(combined_pairs)\n    print(f\"[*] Total combined instruction-response pairs loaded: {total_loaded}\")\n    \n    # Deduplication\n    unique_pairs = []\n    seen_instructions = set()\n    for pair in combined_pairs:\n        # Deduplicate based on instruction content\n        if pair[\"instruction\"] not in seen_instructions:\n            seen_instructions.add(pair[\"instruction\"])\n            unique_pairs.append(pair)\n            \n    total_unique = len(unique_pairs)\n    print(f\"[+] Deduplication complete. Remaining unique records: {total_unique} (Removed {total_loaded - total_unique} duplicates).\")\n    \n    # Deterministic Shuffle for reproducibility\n    print(\"[*] Shuffling dataset deterministically...\")\n    random.seed(42)\n    random.shuffle(unique_pairs)\n    \n    # Split 80 / 10 / 10\n    total = len(unique_pairs)\n    train_end = int(total * 0.8)\n    val_end = train_end + int(total * 0.1)\n    \n    train_split = unique_pairs[:train_end]\n    val_split = unique_pairs[train_end:val_end]\n    test_split = unique_pairs[val_end:]\n    \n    print(f\"\\n[+] Split splits count:\")\n    print(f\"    - Train Split (80%): {len(train_split)} items\")\n    print(f\"    - Val Split (10%): {len(val_split)} items\")\n    print(f\"    - Test Split (10%): {len(test_split)} items\")\n    \n    # Save splits\n    splits = {\n        \"train.json\": train_split,\n        \"val.json\": val_split,\n        \"test.json\": test_split\n    }\n    \n    for filename, split_data in splits.items():\n        output_path = os.path.join(processed_data_dir, filename)\n        with open(output_path, \"w\", encoding=\"utf-8\") as f:\n            json.dump(split_data, f, ensure_ascii=False, indent=2)\n        print(f\"[+] Saved split file: {output_path}\")\n\nif __name__ == \"__main__\":\n    # Clean and split locally if run directly\n    clean_and_split_data()\n";
with open("/content/Retail/src/data_cleaning.py", "w", encoding="utf-8") as f:
    f.write(data_cleaning_code)
print("[+] Created src/data_cleaning.py")

# 4. Write src/verify_models.py
verify_models_code = "import os\nimport json\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import get_peft_model, LoraConfig\n\ndef load_config(config_path):\n    \"\"\"\n    Loads JSON configuration files.\n    \"\"\"\n    if not os.path.exists(config_path):\n        raise FileNotFoundError(f\"[-] Config file not found at {config_path}\")\n    with open(config_path, \"r\", encoding=\"utf-8\") as f:\n        return json.load(f)\n\ndef check_gpu_memory(stage=\"\"):\n    \"\"\"\n    Utility to check and print CUDA memory usage.\n    \"\"\"\n    if torch.cuda.is_available():\n        allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)\n        reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)\n        print(f\"[GPU Memory - {stage}] Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB\")\n    else:\n        print(f\"[Memory - {stage}] CPU RAM in use (CUDA is not available).\")\n\ndef verify_quantized_load(config_path):\n    \"\"\"\n    Loads a model based on its config file in 4-bit quantization and verifies it on CUDA.\n    \"\"\"\n    print(f\"\\n=======================================================\")\n    print(f\"[*] Starting Load Verification for: {config_path}\")\n    print(f\"=======================================================\")\n    \n    # 1. Load config settings\n    config = load_config(config_path)\n    model_name = config.get(\"base_model_name_or_path\")\n    peft_settings = config.get(\"peft_config\", {})\n    quant_settings = config.get(\"quantization_config\", {})\n    \n    print(f\"[+] Base Model: {model_name}\")\n    print(f\"[+] Quantization Config: {json.dumps(quant_settings, indent=2)}\")\n    \n    # 2. Check memory before loading\n    check_gpu_memory(\"Pre-load\")\n    \n    # 3. Setup BitsAndBytes Config\n    # Map compute dtype string to torch dtype\n    compute_dtype_str = quant_settings.get(\"bnb_4bit_compute_dtype\", \"bfloat16\")\n    compute_dtype = torch.bfloat16 if compute_dtype_str == \"bfloat16\" else torch.float16\n    \n    if not torch.cuda.is_available():\n        print(\"[-] ERROR: CUDA is not available. 4-bit quantization requires a GPU. Aborting loading.\")\n        return False\n        \n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=quant_settings.get(\"load_in_4bit\", True),\n        bnb_4bit_quant_type=quant_settings.get(\"bnb_4bit_quant_type\", \"nf4\"),\n        bnb_4bit_use_double_quant=quant_settings.get(\"bnb_4bit_use_double_quant\", True),\n        bnb_4bit_compute_dtype=compute_dtype\n    )\n    \n    # 4. Load tokenizer and model in 4-bit\n    try:\n        print(f\"[*] Loading tokenizer...\")\n        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)\n        # Ensure pad token is configured\n        if tokenizer.pad_token is None:\n            tokenizer.pad_token = tokenizer.eos_token\n            \n        print(f\"[*] Loading model in 4-bit (this can take a few minutes)...\")\n        model = AutoModelForCausalLM.from_pretrained(\n            model_name,\n            quantization_config=bnb_config,\n            device_map=\"auto\",\n            trust_remote_code=True\n        )\n        print(\"[+] Model loaded successfully.\")\n        check_gpu_memory(\"Post-load\")\n        \n        # 5. Apply LoRA Config\n        print(f\"[*] Applying PEFT/LoRA adapter...\")\n        lora_config = LoraConfig(\n            r=peft_settings.get(\"r\", 16),\n            lora_alpha=peft_settings.get(\"lora_alpha\", 32),\n            target_modules=peft_settings.get(\"target_modules\", []),\n            lora_dropout=peft_settings.get(\"lora_dropout\", 0.05),\n            bias=peft_settings.get(\"bias\", \"none\"),\n            task_type=peft_settings.get(\"task_type\", \"CAUSAL_LM\")\n        )\n        \n        model = get_peft_model(model, lora_config)\n        print(\"[+] LoRA Adapter applied successfully.\")\n        \n        # Print trainable parameter percentage\n        model.print_trainable_parameters()\n        \n        # Test basic forward pass dummy generation\n        print(\"[*] Running quick forward pass verification...\")\n        test_input = tokenizer(\"Translate this message: Hello World!\", return_tensors=\"pt\").to(\"cuda\")\n        with torch.no_grad():\n            outputs = model.generate(**test_input, max_new_tokens=10)\n        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)\n        print(f\"[+] Output verified successfully: '{decoded}'\")\n        \n        # Clean memory for the next model\n        del model\n        del tokenizer\n        torch.cuda.empty_cache()\n        print(\"[+] Cleaned up VRAM cache.\")\n        return True\n        \n    except Exception as e:\n        print(f\"[-] ERROR: Failed to load and verify model: {e}\")\n        # Make sure cache is cleared\n        torch.cuda.empty_cache()\n        return False\n\nif __name__ == \"__main__\":\n    # Test loading Qwen or Llama based on argument or default paths\n    qwen_config = \"configs/qwen_lora_config.json\"\n    llama_config = \"configs/llama_lora_config.json\"\n    \n    if os.path.exists(qwen_config):\n        verify_quantized_load(qwen_config)\n    if os.path.exists(llama_config):\n        verify_quantized_load(llama_config)\n";
with open("/content/Retail/src/verify_models.py", "w", encoding="utf-8") as f:
    f.write(verify_models_code)
print("[+] Created src/verify_models.py")

---  
## Step 4: Execute Data Cleaning & Splitting

In [ ]:
# Run the cleaning, merge, and splitting pipeline
import sys
sys.path.append(project_dir)
from src.data_cleaning import clean_and_split_data

clean_and_split_data(raw_data_dir=os.path.join(project_dir, "data/raw"), processed_data_dir=os.path.join(project_dir, "data/processed"))

---  
## Step 5: Quantized Model Loading & LoRA Adapter Verification

In [ ]:
# Execute Qwen load verification (Qwen/Qwen2.5-7B-Instruct)
from src.verify_models import verify_quantized_load

qwen_config_path = os.path.join(project_dir, "configs/qwen_lora_config.json")
verify_quantized_load(qwen_config_path)

In [ ]:
# Execute Llama load verification (meta-llama/Meta-Llama-3-8B-Instruct)
# Note: Accessing Llama models on Hugging Face requires authentication.
# Run !huggingface-cli login in a new cell if you get an access restriction error.

llama_config_path = os.path.join(project_dir, "configs/llama_lora_config.json")
verify_quantized_load(llama_config_path)

---  
## Step 6: Sync Progress Back to Google Drive

In [ ]:
# Check Drive path dynamically for backup
gdrive_target_dir = "/content/drive/MyDrive/Retail"
if os.path.exists("/content/drive/MyDrive/Retail LLM"):
    gdrive_target_dir = "/content/drive/MyDrive/Retail LLM"

print(f"[*] Syncing processed splits and configurations back to Drive: {gdrive_target_dir}...")
os.makedirs(gdrive_target_dir, exist_ok=True)

# Use rsync to backup all new folders, code files, and data splits
!rsync -av --progress /content/Retail/ "{gdrive_target_dir}/"
print("[+] Backup complete. All files successfully persisted to Google Drive!")